In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('../data/processed/merged_data.csv')

In [3]:
df.shape

(590540, 435)

In [4]:
missing_pct  = df.isnull().mean()
missing_pct.head()

TransactionID     0.0
isFraud           0.0
TransactionDT     0.0
TransactionAmt    0.0
ProductCD         0.0
dtype: float64

In [5]:
cols_to_drop =  missing_pct[missing_pct > 0.7].index
len(cols_to_drop)

208

In [6]:
df.drop(cols_to_drop, axis=1, inplace=True)
df.shape

(590540, 227)

Step 1 — Drop High Missing Value Columns

- Original dataset: 435 columns
- Threshold: drop any column with more than 70% missing values
- Columns dropped: 208
- Remaining columns: 227
- Dropping these columns removes noise and speeds up model training
- A column that is 70%+ empty cannot reliably teach the model anything

In [7]:
df.isnull().sum().sum()

np.int64(16088043)

In [8]:
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[numerical_cols] = df[numerical_cols].fillna(df[numerical_cols].median())

In [9]:
df.isnull().sum().sum()

np.int64(2750959)

In [10]:
categorical_cols = df.select_dtypes(include=['object']).columns
df[categorical_cols] = df[categorical_cols].fillna('Unknown')

df.isnull().sum().sum()

## Step 2 — Fill Missing Values

- Started with 16,088,043 missing values across 227 columns
- Numerical columns: filled with median value of each column
  - Median used instead of mean because outliers like $31,937 
    transactions would skew the mean significantly
- Categorical columns: filled with 'Unknown'
  - Unknown tells the model that information was not available
  - This is itself a signal — missing identity info can indicate fraud
- Result: 0 missing values remaining
- Dataset is now complete and ready for encoding

In [13]:
df[categorical_cols].nunique()


ProductCD         5
card4             5
card6             5
P_emaildomain    60
M1                3
M2                3
M3                3
M4                4
M5                3
M6                3
M7                3
M8                3
M9                3
dtype: int64

In [14]:
from sklearn.preprocessing import LabelEncoder

In [15]:
le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

In [16]:
df[categorical_cols].head()

,ProductCD,card4,card6,P_emaildomain,M1,M2,M3,M4,M5,M6,M7,M8,M9
0,4,2,2,0,1,1,1,2,0,1,2,2,2
1,4,3,2,17,2,2,2,0,1,1,2,2,2
2,4,4,3,36,1,1,1,0,0,0,0,0,0
3,4,3,3,54,2,2,2,0,1,0,2,2,2
4,1,3,2,17,2,2,2,3,2,2,2,2,2


## Step 3 — Encode Categorical Columns

- 13 categorical columns identified
- All columns encoded using Label Encoding
- Label encoding converts each unique text value to a number
- Example: Visa=0, Mastercard=1, Discover=2, Amex=3
- One-hot encoding was not used because P_emaildomain has 60 unique 
  values which would add 60 new columns — too many
- Result: all columns are now numerical and ready for model training

In [17]:
cols_to_scale = numerical_cols.drop(['isFraud', 'TransactionID'])


In [18]:
from sklearn.preprocessing import StandardScaler


In [19]:
scaler = StandardScaler()
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

In [20]:
df['TransactionAmt'].mean()
df['TransactionAmt'].std()

np.float64(1.0000008466837726)

In [21]:
df['TransactionAmt'].mean()

np.float64(5.914972887521002e-17)

In [22]:
df['TransactionAmt'].std()

np.float64(1.0000008466837726)

## Step 4 — Scale Numerical Columns

- Problem: columns had very different ranges
  - TransactionAmt: $0.25 to $31,937
  - hour: 0 to 24
  - Large range differences cause models to incorrectly weight columns
- Solution: StandardScaler applied to all numerical columns
- StandardScaler converts each column to mean=0 and std=1
- Verified: TransactionAmt mean is now ~0 and std is now ~1
- isFraud and TransactionID were excluded from scaling
  - Never scale your target variable
  - Never scale your ID column

In [24]:
df.to_csv('../data/processed/cleaned_data.csv', index=False)